In [ ]:
!mkdir ~/tmpdir
!TMPDIR=~/tmpdir python3 -m pip install --no-cache-dir --upgrade "sagemaker<3" boto3 pandas numpy scikit-learn

In [ ]:
import sagemaker
import boto3
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [ ]:
sess = sagemaker.Session()
region = sess.boto_region_name

bucket = sess.default_bucket()
role = sagemaker.get_execution_role()

prefix = "sagemaker"
print(bucket, region)

In [ ]:
from sagemaker import image_uris

container = sagemaker.image_uris.retrieve(
    framework="xgboost", region=region, version="latest"
)

In [ ]:
xgb = sagemaker.estimator.Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{bucket}/{prefix}/model",
    sagemaker_session=session,
)

xgb.set_hyperparameters(
    objective="multi:softmax",
    num_class=3,
    num_round=50,
    max_depth=4,
    eta=0.1,
)

In [ ]:
train_input = sagemaker.inputs.TrainingInput(<DATASET>, content_type="text/csv")
xgb.fit({"train": train_input})